# Imports

In [ ]:
import numpy as np
import torch
from torch import Tensor
import matplotlib.pyplot as plt
from functools import partial
import seaborn as sns
import sys
import os
import einops
import math

import transformer_lens
from transformer_lens import HookedTransformer
from transformer_lens.hook_points import HookPoint
import transformer_lens.utils as utils

torch.set_grad_enabled(False)
print("Disabled automatic differentiation")

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

%load_ext autoreload
%autoreload 2

# Setup

In [ ]:
model = HookedTransformer.from_pretrained(
    "meta-llama/Llama-3.2-1B",  # rope_freq = 500000
    # "phi-1",   # rope_freq = 10000
    # "attn-only-2l",
    # "attn-only-4l",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=False,
    default_prepend_bos=False
)
device: torch.device = utils.get_device()

model = model.to(device)                              

In [ ]:
# this removes padding bos, annoying at visualizing attention patterns
model.cfg.default_prepend_bos = False
model.cfg.tokenizer_prepends_bos = False

# expands GQA heads for n_kv_heads == n_heads
model.set_ungroup_grouped_query_attention(True) 
# assert model.cfg.n_key_value_heads == model.cfg.n_heads, f"n_kv_heads must be equal to n_heads for this model, got  {model.cfg.n_key_value_heads}, {model.cfg.n_heads}"
model.W_V.shape, model.W_K.shape 

In [ ]:
with open('../data/tiny_shakespeare.txt', 'r') as f:
    lines = f.readlines()
    
lines = [line.strip() for line in lines if line.strip()]  # Remove empty lines
lines = [line.replace('\n', '') for line in lines]  # Remove newline characters
lines = lines[:model.cfg.n_ctx]

print("Length of lines:", len(lines))
text = " ".join(lines)
# insertions = {
#     1: "The ",
#     20: "\n",
#     40: "\n",
#     60: "\n"
# }

# modified_prompt = insert_tokens_at_positions(lines, insertions)
# print(modified_prompt)

In [ ]:
# some needle in a haystack
lines_new = lines[:200]  
text_new = " ".join(lines_new)
text_new = text_new + "Who is the enemy of the state?" 
answer = "Caius Marcius"

utils.test_prompt(text_new, answer, model, prepend_space_to_answer=False, print_details=False)

model.set_ungroup_grouped_query_attention(False)  # else errors
model.generate(
    text_new,
    max_new_tokens=20,
    temperature=2.5,
    top_k=2,
    top_p=1.0,
    do_sample=False,
)

In [ ]:
from data.succession import generate_successor_pairs, create_prompt, create_flipped_prompt, create_augmented_prompts

succession_dataset, succession_mapping = generate_successor_pairs()

task_prompts = create_prompt(succession_dataset)
flipped_task_prompts = create_flipped_prompt(succession_dataset)

aug_data = create_augmented_prompts(task_prompts, flipped_task_prompts)

create_single_prompt_lambda = lambda task_prompts: " ".join(task_prompts.values())
prompts = create_single_prompt_lambda(task_prompts)

## This prints just the sequence
# for task_name, task_prompt in task_prompts.items():
    # print(f"{task_name}: {task_prompt}")

aug_clean_prompts = [aug["clean"] for aug in aug_data]
aug_flipped_prompts = [aug["corrupt"] for aug in aug_data]
for i, (task_name, task_prompt) in enumerate(zip(task_prompts, flipped_task_prompts)):
    print(aug_data[i]["clean"])

In [ ]:
# aug_prompt = f"The next number in the sequence {task_prompts['Numbers']} is "
# aug_prompt = f"The last number in the sequence {task_prompts['Numbers']} is"
# aug_prompt

# run a forward pass and cache intermediate activations from Hook Points (check using model.hook_dict)
tokens = model.to_tokens(aug_clean_prompts, prepend_bos=False)
# tokens = tokens[:, :model.cfg.n_ctx//2]  # Truncate if longer than n_ctx
print(tokens.shape)
# Define a global cache to collect all layer activations
all_cache = {}

# Define the hook function
def hook_fn(activation: torch.Tensor, hook: HookPoint, name: str = ""): # removing, the hook arg weirdly breaks
    all_cache[name] = activation
    return activation

# Collect rope related activations in each layer
fwd_hooks = []
for layer in range(model.cfg.n_layers):
    for hook_name in [
        f"blocks.{layer}.attn.hook_q", # pre-rope
        f"blocks.{layer}.attn.hook_k", # pre-rope
        f"blocks.{layer}.attn.hook_rot_q", # post-rope
        f"blocks.{layer}.attn.hook_rot_k", # post-rope
        f"blocks.{layer}.attn.hook_attn_scores", # pre-softmax
        f"blocks.{layer}.attn.hook_pattern", # post-softmax
    ]:
        fwd_hooks.append((hook_name, partial(hook_fn, name=hook_name)))

model.set_ungroup_grouped_query_attention(True)
# Run the model ONCE with all hooks active
# 1 GB difference in memory usage, between *run_with_hooks* and *run_with_cache*
_ = model.run_with_hooks(tokens, fwd_hooks=fwd_hooks)
# all_logits, all_cache = model.run_with_cache(tokens)

# Check if the cache stored all necessary acts
for n, p in all_cache.items():
    print(n)

In [ ]:
from utils.rope_features import get_all_rope_acts

# Get the rope activations
rot_act_q_layers, rot_act_k_layers, pre_rot_act_q_layers, pre_rot_act_k_layers = get_all_rope_acts(model, all_cache)

# Stack the collected tensors along a new dimension
rot_act_q_stacked = torch.stack(rot_act_q_layers, dim=0)
rot_act_k_stacked = torch.stack(rot_act_k_layers, dim=0)
pre_rot_act_q_stacked = torch.stack(pre_rot_act_q_layers, dim=0)
pre_rot_act_k_stacked = torch.stack(pre_rot_act_k_layers, dim=0)

print(rot_act_q_stacked.shape) # n_layers, bsz, seq_len, n_heads, head_dim
print(rot_act_k_stacked.shape) # n_layers, bsz, seq_len, n_heads, head_dim
print(pre_rot_act_q_stacked.shape) # n_layers, bsz, seq_len, n_heads, head_dim
print(pre_rot_act_k_stacked.shape) # n_layers, bsz, seq_len, n_heads, head_dim

In [ ]:
print(pre_rot_act_q_stacked.mean(), rot_act_q_stacked.std())
print(rot_act_q_stacked.mean(), rot_act_q_stacked.std())

print(pre_rot_act_k_stacked.mean(), rot_act_k_stacked.std())
print(rot_act_k_stacked.mean(), rot_act_k_stacked.std())

In [ ]:
from utils.rope_features import plot_rope_freq_per_head_layer

plot_rope_freq_per_head_layer(rot_act_q_stacked.cpu().numpy(), rot_act_k_stacked.cpu().numpy(), mode='layers', layer_to_plot=None)
plot_rope_freq_per_head_layer(rot_act_q_stacked.cpu().numpy(), rot_act_k_stacked.cpu().numpy(), mode='heads', layer_to_plot=None)
plot_rope_freq_per_head_layer(rot_act_q_stacked.cpu().numpy(), rot_act_k_stacked.cpu().numpy(), mode='per_layer', layer_to_plot=0)

In [ ]:
n_layers, bsz, seq_len, n_heads, head_dim = pre_rot_act_q_stacked.shape
n_freqs = head_dim // 2
device = pre_rot_act_q_stacked.device
positions = torch.arange(50, device=device)
theta_i = 2 * np.pi * torch.arange(n_freqs, device=device) / head_dim

# Step 1: Reshape into 2D rotary pairs: (..., n_freqs, 2)
q_freq = pre_rot_act_q_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)
k_freq = pre_rot_act_k_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)

# Step 2: Compute the mean vector across all positions (layer, batch, seq, head)
q_mean_2d = q_freq.mean(dim=(0, 1, 2, 3))  # (n_freqs, 2)
k_mean_2d = k_freq.mean(dim=(0, 1, 2, 3))  # (n_freqs, 2)

# Step 3: Compute L2 norm of the mean vectors (1 value per rotary pair)
q_norm = q_mean_2d.norm(dim=-1)  # (n_freqs,)
k_norm = k_mean_2d.norm(dim=-1)  # (n_freqs,)

# Step 4: Compute angle φᵢ between mean query and key vector per freq
dot = (q_mean_2d * k_mean_2d).sum(dim=-1)
cos_phi = torch.clamp(dot / (q_norm * k_norm + 1e-6), -1, 1)
phi = torch.acos(cos_phi)  # (n_freqs,)

# Step 5: Build dᵢ(p) for each rotary pair i
d_p = []
for i in range(n_freqs):
    d_i = q_norm[i] * k_norm[i] * torch.cos(phi[i] + theta_i[i] * positions)
    d_p.append(d_i)
d_p = torch.stack(d_p, dim=0)  # (n_freqs, seq_len)
D_p = d_p.sum(dim=0)           # (seq_len,)

# Step 6: Plot
plt.figure(figsize=(12, 6))
for i in range(n_freqs//2):
    plt.plot(positions.cpu().numpy(), d_p[i].cpu().numpy(), label=f"Feature {i}")
plt.plot(positions.cpu().numpy(), D_p.cpu().numpy(), label="Sum D(p)", color="black", linewidth=2)
plt.xlabel("Position p")
plt.ylabel("Dot product $d_i(p)$")
plt.title("L2 norm of mean rotary vectors and resulting $d_i(p)$ curves")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(True, linestyle='--', linewidth=0.5)
# plt.xticks(ticks=np.arange(0, seq_len, step=4), labels=[str(i) for i in range(0, seq_len, 4)])
plt.tight_layout()
plt.show()

In [ ]:
seq_len = D_p.shape[0]
d_head = head_dim 

# Step 1: Create D(p - t) → matrix D[i, j] = D(i - j)
position_diffs = torch.arange(seq_len).unsqueeze(1) - torch.arange(seq_len).unsqueeze(0)  # (seq_len, seq_len)
D_matrix = D_p[(position_diffs + seq_len) % seq_len]  # wrap-around indexing

# Step 2: Scale by 1/√d_head
D_scaled = D_matrix / np.sqrt(d_head)

# Step 3: Apply causal mask
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
D_scaled[mask] = float('-inf')

# Step 4: Apply softmax across rows to get attention pattern
attention_pattern = torch.nn.functional.softmax(D_scaled, dim=-1)

# decoded_tokens = model.to_str_tokens(tokens[0])  # for a single input sequence

# Step 5: Plot the attention pattern
plt.figure(figsize=(8, 6))
plt.imshow(attention_pattern.cpu().numpy(), cmap="viridis", aspect='auto')
plt.colorbar(label="Attention Score")
# plt.xticks(ticks=range(len(decoded_tokens)), labels=decoded_tokens, rotation=80)
# plt.yticks(ticks=range(len(decoded_tokens)), labels=decoded_tokens)
plt.xlabel("Key position $n$")
plt.ylabel("Query position $m$")
plt.title("Reconstructed Positional Attention Pattern from Rotary Dot Products")
plt.tight_layout()
plt.show()

In [ ]:
# — your feature/index setup —
rope_feature = 10
layer_idx = 0
head_idx  = 2

assert 0 <= rope_feature < n_freqs

n_layers, bsz, seq_len, n_heads, head_dim = rot_act_q_stacked.shape
assert head_dim % 2 == 0, "head_dim must be even so you can split into sin/cos pairs"
n_freqs = head_dim // 2

# — reshape RoPE’d activations —
rot_q = pre_rot_act_q_stacked.contiguous().view(
    n_layers, bsz, seq_len, n_heads, n_freqs, 2
)
rot_k = pre_rot_act_k_stacked.contiguous().view(
    n_layers, bsz, seq_len, n_heads, n_freqs, 2
)

# — extract the 2D coords for this layer/head/feature —
q2d = rot_q[layer_idx, 0, :, head_idx, rope_feature, :]   # (seq_len,2)
k2d = rot_k[layer_idx, 0, :, head_idx, rope_feature, :]   # (seq_len,2)

# — compute per‐position means —
mean_q = q2d.mean(dim=0)   # tensor([x, y])
mean_k = k2d.mean(dim=0)

# — numpy versions for plotting —
q_np    = q2d.cpu().numpy()
k_np    = k2d.cpu().numpy()
mean_qn = mean_q.cpu().numpy()
mean_kn = mean_k.cpu().numpy()

# — compute enclosing circle radius R (with extra padding) —
all_pts = np.vstack([q_np, k_np])
r_max   = np.linalg.norm(all_pts, axis=1).max()
R       = r_max * 1.02

# — compute RoPE inverse‐frequency ω for feature‐pair i//2 —
d_head = head_dim
base   = model.cfg.rotary_base
k      = rope_feature 
omega_rad = base ** (-2.0 * i / d_head)  # radians per step
omega_deg = np.degrees(omega_rad)        # degrees per step
print(f"Frequency (ω) = {omega_rad:.2e} radians/token or {np.degrees(omega_rad):.2e} degrees/token")
period = 2 * np.pi / omega_rad
print(f"Full rotation period ≈ {period:.1f} tokens")
print(f"Full rotation period ≈ {period/model.cfg.n_ctx:.1%} of the sequence length")
# — compute the final rotated mean‐query after seq_len steps —
angle_final = seq_len * omega_rad
print(f"Final angle (θ) = {angle_final:.2f} radians or {np.degrees(angle_final):.2f} degrees or {angle_final/np.pi:.2f} π or {angle_final / (2*np.pi):.2f} full rotations")

c, s = np.cos(angle_final), np.sin(angle_final)
rotated_mean_q = np.array([
    c * mean_qn[0] - s * mean_qn[1],
    s * mean_qn[0] + c * mean_qn[1]
])

# choose a desire extension length (in the same units as your plot axes)
# d = 1.0
# # direction vector and its unit version
# v    = rotated_mean_q - mean_qn
# u    = v / np.linalg.norm(v)
# # new head is pushed out by d
# arrow_head = rotated_mean_q + d * u

# k_extra    = 12.0
# angle_ext  = (seq_len + k_extra) * omega_rad

# # rotate mean‐Q by this extended angle
# c, s = np.cos(angle_ext), np.sin(angle_ext)
# rotated_q_ext = np.array([
#     c * mean_qn[0] - s * mean_qn[1],
#     s * mean_qn[0] + c * mean_qn[1]
# ])

# # then in your plotting section, replace
# #   rotated_mean_q   # ← your old end‐point
# # with
# arrow_head = rotated_q_ext

# new tail is pushed out by d
# — begin plotting —
fig, ax = plt.subplots()

# big enclosing circle
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(R * np.cos(theta),
        R * np.sin(theta),
        color='gray', alpha=0.4)

# scatter data & means
ax.scatter(q_np[:,0], q_np[:,1], color='lightcoral', alpha=0.3, label='Query')
ax.scatter(k_np[:,0], k_np[:,1], color='lightblue', alpha=0.3, label='Key')
ax.scatter(*mean_qn, color='red',   s=80, label='Mean Query')
ax.scatter(*mean_kn, color='green', s=80, label='Mean Key')
ax.scatter(0, 0,         color='black', s=40, label='Origin')

# curved arrow just from mean_q to its rotated position
ax.annotate(
    "",
    xy=tuple(rotated_mean_q),   # the rotated mean-Q
    # xy=tuple(arrow_head),   # the rotated mean-Q with extended point, not needed
    xytext=tuple(mean_qn),      # the original mean-Q
    arrowprops=dict(
        arrowstyle='-|>',
        color='purple',
        lw=2,
        connectionstyle='arc3,rad=0.3'
    )
)

# keep tick labels for x & y
ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

# re-center spines at the origin
ax.spines['left'].set_position(('data', 0))
ax.spines['bottom'].set_position(('data', 0))
for loc in ['right','top']:
    ax.spines[loc].set_color('none')

# equal aspect & limits
ax.set_aspect('equal', 'box')
ax.set_xlim(-R, R)
ax.set_ylim(-R, R)

# labels, grid, legend
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(linestyle='--', alpha=0.3)
ax.legend(loc='upper left', bbox_to_anchor=(1,1))
ax.set_title(f"RoPE Rotation (layer={layer_idx}, head={head_idx}, feat={rope_feature})")

plt.tight_layout()
plt.show()

In [ ]:
# pick one feature-pair i in [0..n_freqs-1]
i = 10

# reshape RoPE’d activations into (L, B, S, H, F, 2)
n_layers, bsz, seq_len, n_heads, head_dim = rot_act_q_stacked.shape
n_freqs = head_dim // 2
rot_q = rot_act_q_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)
rot_k = rot_act_k_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)

# extract per-position 2D and the *mean* vectors
q2d    = rot_q[layer_idx, 0, :, head_idx, i, :]       # (S,2)
mean_q = q2d.mean(dim=0)                              # (2,)
mean_k = rot_k[layer_idx, 0, :, head_idx, i, :].mean(dim=0)  # (2,)

# norms and signed phase offset φ
norm_q = mean_q.norm()
norm_k = mean_k.norm()
alpha_q = torch.atan2(mean_q[1], mean_q[0])
alpha_k = torch.atan2(mean_k[1], mean_k[0])
phi     = (alpha_q - alpha_k + np.pi) % (2*np.pi) - np.pi

# RoPE inverse-frequency ω_i
d_head = head_dim
base   = model.cfg.rotary_base
omega_rad = base ** (-2.0 * i / d_head)  # radians per step
omega_deg = np.degrees(omega_rad)        # degrees per step
print(f"Frequency (ω) = {omega_rad:.2e} radians/token or {np.degrees(omega_rad):.2e} degrees/token")
period = 2 * np.pi / omega_rad
print(f"Full rotation period ≈ {period:.1f} tokens")
print(f"Full rotation period ≈ {period/model.cfg.n_ctx:.1%} of the sequence length")
print(f"Final angle (θ) = {angle_final:.2f} radians or {np.degrees(angle_final):.2f} degrees or {angle_final/np.pi:.2f} π or {angle_final / (2*np.pi):.2f} full rotations")

# build positions & angles
positions   = torch.arange(seq_len, dtype=mean_q.dtype, device=mean_q.device)
angles      = omega_rad * positions   # (S,)

# now both “predicted” and “actual” use the *rotated mean_q*
# predicted via closed-form:
dot_pred = (norm_q * norm_k) * torch.cos(phi + angles)

# actual by rotating mean_q explicitly and dotting with mean_k
c = angles.cos().unsqueeze(1)  # (S,1)
s = angles.sin().unsqueeze(1)
# build rotation matrices implicitly and apply to mean_q
qs = torch.cat([c*mean_q[0] - s*mean_q[1],
                s*mean_q[0] + c*mean_q[1]], dim=1)  # (S,2)
dot_actual = (qs * mean_k).sum(dim=1)                 # (S,)

# plot
plt.figure(figsize=(6,3))
plt.plot(positions.cpu(), dot_actual.cpu(), label='Actual $d_i(p)$')
plt.plot(positions.cpu(), dot_pred.cpu(), '--', label='Predicted $\hat d_i(p)$')
plt.legend()
plt.xlabel('Position $p$')
plt.ylabel(f'Feature‐pair {i} dot‐product')
plt.title(f'Actual vs Predicted RoPE dot‐product (feat {i})')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# error check
max_err = (dot_actual - dot_pred).abs().max().item()
print(f"max abs error over positions = {max_err:.2e}")

In [ ]:
from utils.detect_head import detect_head, get_supported_heads
from utils.plot_head import imshow

aug_prompt = f"The next number in the sequence {task_prompts['Numbers']} is"
utils.test_prompt(aug_prompt, answer="20", model=model, prepend_space_to_answer=True, print_details=False)
number_toks = model.to_tokens(aug_prompt, prepend_bos=False)

model.reset_hooks()
model.set_ungroup_grouped_query_attention(True)
_, cache = model.run_with_cache(
    # text[:100],
    number_toks,
    )

get_supported_heads()
previous_token_heads_scores = detect_head(model, seq=aug_clean_prompts, detection_pattern='previous_token_head', cache=all_cache, error_measure='mul')

def plot_head_detection_scores(
    scores: torch.Tensor,
    zmin: float = -1,
    zmax: float = 1,
    xaxis: str = "Head",
    yaxis: str = "Layer",
    title: str = "Head Matches"
) -> None:
    imshow(scores, zmin=zmin, zmax=zmax, xaxis=xaxis, yaxis=yaxis, title=title)

plot_head_detection_scores(previous_token_heads_scores, title="Previous Token Head Matches")

In [ ]:
from utils.plot_head import show_attention_patterns

## long range test
# highlighted = ["Caius", "Marcius", "enemy", " enemy", "state", " state", "?"]
# model.to_str_tokens(model.to_tokens("Caius Marcius"))
# lines_new = lines[:10] 
# text_new = " ".join(lines_new)
# text_new = text_new + "Who is the enemy of the state?" 

# show_attention_patterns(model, [(0, 0)], prompts=text_new, mode="pattern", highlight_words=highlighted, show_all_tokens=False, return_fig=True) 
show_attention_patterns(model, [(0, 2)], prompts=aug_clean_prompts[0], mode="pattern", show_all_tokens=True, return_fig=True) 

In [ ]:
from utils.rope_features import extract_rope_frequency_usage

layer = 0
head = 2

q_rot_head = rot_act_q_stacked[layer, 0, :, head, :]
k_rot_head = rot_act_k_stacked[layer, 0, :, head, :]

extract_rope_frequency_usage(
    model,
    prompt=aug_prompt,
    q_rot=q_rot_head,
    k_rot=k_rot_head,
    layer_idx=layer,
    head_idx=head,
    q_or_k="q",
    show_freqs=True
)

extract_rope_frequency_usage(
    model,
    prompt=aug_prompt,
    q_rot=q_rot_head,
    k_rot=k_rot_head,
    layer_idx=layer,
    head_idx=head,
    q_or_k="k",
    show_freqs=True
)

In [ ]:
layer_idx = 0
head_idx  = 2
i         = 10          # feature-pair index 0..(head_dim/2-1)

n_layers, bsz, seq_len, n_heads, head_dim = rot_act_q_stacked.shape
assert head_dim % 2 == 0
n_freqs = head_dim // 2
d_head  = head_dim

# reshape into 2-D sin/cos pairs
rot_q = rot_act_q_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)
rot_k = rot_act_k_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)

seq_len = 100
# extract per-position queries and mean key for this feature
q2d    = rot_q[layer_idx, 0, :, head_idx, i, :]       # (seq_len, 2)
mean_q = q2d.mean(dim=0)                              # (2,)
mean_k = rot_k[layer_idx, 0, :, head_idx, i, :].mean(dim=0)  # (2,)

# norms
norm_q = mean_q.norm()
norm_k = mean_k.norm()

# 1) signed phase offset φ = arg(k) − arg(q)
α_q = torch.atan2(mean_q[1], mean_q[0])
α_k = torch.atan2(mean_k[1], mean_k[0])
φ   = α_k - α_q
# normalize into (−π, π]:
φ = (φ + np.pi) % (2*np.pi) - np.pi

# 2) RoPE inverse-frequency ω for this pair
base    = 50000.0
ω       = base ** (-2.0 * i / d_head)     # radians per token

# 3) build the per-position d_i(p)
positions = torch.arange(seq_len, dtype=mean_q.dtype, device=mean_q.device)
d_ip      = (norm_q * norm_k) * torch.cos(φ + ω * positions)  # shape (seq_len,)

# 4) toy “attention” matrix from this single feature
scale = np.sqrt(d_head)
attn = torch.full((seq_len, seq_len), float('-inf'), device=mean_q.device)
for p in range(seq_len):
    # only allow keys ≤ p (autoregressive mask)
    scores = d_ip / scale    # same score for all keys at this q, 
                             # but you could also compare q2d[p] vs k2d[:p+1]
    attn[p, :p+1] = scores[:p+1]

# apply softmax
attn = torch.softmax(attn, dim=-1)  # (seq_len, seq_len)

# 5) test your bounds
bound1 = (ω < 2*np.pi/seq_len)
bound2 = (φ > np.pi + (seq_len*ω)/2)

# 6) test offset-criterion: d_i(p) < d_i(0) for all p ≤ pmax
offset_ok = bool((d_ip <= d_ip[0]).all())

print(f"ω = {ω:.3e}, bound1 = ω < 2π/seq_len → {bound1}")
print(f"φ = {φ:.3f}, bound2 = φ > π + (seq_len*ω)/2 → {bound2}")
print(f"offset_criterion (d_i(p)<d_i(0) ∀p) → {offset_ok}")

# — visualize —
fig, axs = plt.subplots(1,2,figsize=(10,4))

# dot-product curve
axs[0].plot(positions.cpu(), d_ip.cpu())
axs[0].axhline(d_ip[0].cpu(), linestyle='--', color='gray')
axs[0].set_title(f"Feature {i}: d_i(p)")
axs[0].set_xlabel("p"); axs[0].set_ylabel("dot-product")

# attention heatmap
im = axs[1].imshow(attn.cpu().numpy(), cmap='viridis')
axs[1].set_title(f"Attention from feature {i}")
axs[1].set_xlabel("key position"); axs[1].set_ylabel("query position")
fig.colorbar(im, ax=axs[1], label="attention weight")

plt.tight_layout()
plt.show()

In [ ]:
# — hyperparams and indices —
layer_idx = 0
head_idx  = 2
base      = 50000.0

# your RoPE’d Q/K stacks: (L, B, seq_len, H, head_dim)
# head_dim must be even
n_layers, bsz, seq_len, n_heads, head_dim = rot_act_q_stacked.shape
assert head_dim % 2 == 0
n_freqs   = head_dim // 2
d_head    = head_dim
scale     = np.sqrt(d_head)

# reshape into pairs and flatten back to head_dim
rot_q = rot_act_q_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)
rot_k = rot_act_k_stacked.view(n_layers, bsz, seq_len, n_heads, n_freqs, 2)
# flatten the last two dims → (L, B, seq_len, H, head_dim)
rot_q_flat = rot_q.view(n_layers, bsz, seq_len, n_heads, head_dim)
rot_k_flat = rot_k.view(n_layers, bsz, seq_len, n_heads, head_dim)

# pick out Q & K for our single layer/batch/head
Q = rot_q_flat[layer_idx, 0, :, head_idx, :]   # (seq_len, head_dim)
K = rot_k_flat[layer_idx, 0, :, head_idx, :]   # (seq_len, head_dim)

# --- Step 1: find your “offset” feature‐pairs i ---
# compute ω_i and φ_i for all i in [0..n_freqs-1]
i_idx     = torch.arange(n_freqs, dtype=torch.float32)
ω_i       = base ** (-2.0 * i_idx / d_head)             # (n_freqs,)
# compute φ_i from the *mean* over positions
# mean_q_i and mean_k_i are the center of each pair
mean_q_pairs = rot_q[layer_idx, 0, :, head_idx].mean(dim=0)  # (n_freqs,2)
mean_k_pairs = rot_k[layer_idx, 0, :, head_idx].mean(dim=0)
α_q         = torch.atan2(mean_q_pairs[:,1], mean_q_pairs[:,0])  # (n_freqs,)
α_k         = torch.atan2(mean_k_pairs[:,1], mean_k_pairs[:,0])
φ_i         = α_k - α_q
φ_i         = (φ_i + np.pi) % (2*np.pi) - np.pi

# your two bounds
bound1 = ω_i < (2*np.pi/seq_len)
bound2 = φ_i > (np.pi + (seq_len*ω_i.to(φ_i.device))/2)

# offset‐feature mask
offset_mask = bound1.to(φ_i.device) & bound2
offset_ids  = torch.nonzero(offset_mask).flatten().tolist()

print("Offset feature‐pairs:", offset_ids)

# convert those pair‐ids to head_dim indices: each pair i maps to dims [2i, 2i+1]
offset_dims = []
for j in offset_ids:
    offset_dims += [2*j, 2*j+1]

# build a head_dim‐mask
mask_vec = torch.zeros(head_dim, device=Q.device)
mask_vec[offset_dims] = 1.0

# --- Step 2: full vs partial attention scores ---
# compute full scaled dot‐product scores
scores_full = (Q @ K.T) / scale    # (seq_len, seq_len)
# compute partial (offset‐only) scores
Q_off  = Q * mask_vec.unsqueeze(0)  # zero out non-offset dims
K_off  = K * mask_vec.unsqueeze(0)
scores_off  = (Q_off @ K_off.T) / scale

# apply causal mask
causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=Q.device), diagonal=1).bool()
scores_full.masked_fill_(causal_mask, float('-inf'))
scores_off.masked_fill_(causal_mask, float('-inf'))

# Step 3: softmax to get probabilities
attn_full = torch.softmax(scores_full, dim=-1)
attn_off  = torch.softmax(scores_off,  dim=-1)

# — visualize both heatmaps —
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(10,4), sharey=True)
im1 = ax1.imshow(attn_full.cpu(), cmap='viridis')
ax1.set_title("Full head attention")
ax1.set_xlabel("key pos")
ax1.set_ylabel("query pos")
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

im2 = ax2.imshow(attn_off.cpu(), cmap='viridis')
ax2.set_title(f"Offset‐only attention\n({len(offset_ids)} pairs)")
ax2.set_xlabel("key pos")
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
# — hyperparams & tensor shapes —
layer_idx = 0
head_idx  = 2
base      = 50000.0

L, B, seq_len, H, head_dim = rot_act_q_stacked.shape
assert head_dim % 2 == 0
n_freqs   = head_dim // 2
scale     = np.sqrt(head_dim)

# reshape Q/K into (L, B, seq_len, H, n_freqs, 2) then flatten back
rot_q = rot_act_q_stacked.view(L, B, seq_len, H, n_freqs, 2)
rot_k = rot_act_k_stacked.view(L, B, seq_len, H, n_freqs, 2)
rot_q = rot_q.view(L, B, seq_len, H, head_dim)
rot_k = rot_k.view(L, B, seq_len, H, head_dim)

# pick out our single layer/batch/head
Q_all = rot_q [layer_idx, 0, :, head_idx, :]   # (seq_len, head_dim)
K_all = rot_k [layer_idx, 0, :, head_idx, :]

# compute per-pair means μ_q,i, μ_k,i
Q_pairs = rot_act_q_stacked.view(L, B, seq_len, H, n_freqs, 2)
K_pairs = rot_act_k_stacked.view(L, B, seq_len, H, n_freqs, 2)
μq = Q_pairs[layer_idx, 0, :, head_idx].mean(dim=0)  # (n_freqs, 2)
μk = K_pairs[layer_idx, 0, :, head_idx].mean(dim=0)

# norms
norm_q = μq.norm(dim=1)   # (n_freqs,)
norm_k = μk.norm(dim=1)

# inverse freqs ω_i
i_idx = torch.arange(n_freqs, dtype=torch.float32, device=rot_act_q_stacked.device)
ω_i   = base ** (-2.0 * i_idx / head_dim)          # (n_freqs,)

# positions
p = torch.arange(seq_len, dtype=torch.float32, device=rot_act_q_stacked.device).unsqueeze(1)  # (seq_len,1)

# compute all angles = p * ω_i  → shape (seq_len, n_freqs)
angles = p * ω_i.unsqueeze(0)

# compute signed φ_i = arg(μk)-arg(μq)
αq = torch.atan2(μq[:,1], μq[:,0])   # (n_freqs,)
αk = torch.atan2(μk[:,1], μk[:,0])
φ  = αk - αq
φ  = (φ + np.pi) % (2*np.pi) - np.pi  # normalize

# compute d_i(p) = ‖μq‖‖μk‖ cos(φ_i + ω_i p)
# first broadcast norms and φ
R  = norm_q * norm_k                       # (n_freqs,)
Φp = φ.unsqueeze(0) + angles               # (seq_len, n_freqs)
D  = R.unsqueeze(0) * torch.cos(Φp)        # (seq_len, n_freqs)

# now pick offset‐features = those i for which D[p, i] < D[0, i] ∀ p>0
# exclude p=0 itself
mask = (D[1:] < D[0:1]).all(dim=0)         # (n_freqs,) boolean
offset_ids = torch.nonzero(mask).flatten().tolist()
print("Offset feature‐pairs:", offset_ids)

# build head_dim mask for those i
offset_dims = [d for i in offset_ids for d in (2*i, 2*i+1)]
mask_vec    = torch.zeros(head_dim, device=Q_all.device)
mask_vec[offset_dims] = 1.0

# --- reconstruct full vs offset‐only attention ---

# full scaled‐dot scores
scores_full = (Q_all @ K_all.T) / scale
# offset‐only scores
Q_off = Q_all * mask_vec.unsqueeze(0)
K_off = K_all * mask_vec.unsqueeze(0)
scores_off = (Q_off @ K_off.T) / scale

# causal mask
causal = torch.triu(torch.ones(seq_len, seq_len, device=Q_all.device), 1).bool()
scores_full.masked_fill_(causal, -1e9)
scores_off.masked_fill_(causal, -1e9)

# softmax
attn_full = torch.softmax(scores_full, dim=-1)
attn_off  = torch.softmax(scores_off,  dim=-1)

# plot
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(10,4), sharey=True)
ax1.imshow(attn_full.cpu(), cmap='viridis')
ax1.set(title='Full head attention', xlabel='key pos', ylabel='query pos')
ax2.imshow(attn_off.cpu(), cmap='viridis')
ax2.set(title=f'Offset‐only attention\n({len(offset_ids)} pairs)', xlabel='key pos')
plt.tight_layout()
plt.show()

In [ ]:
from utils.patching_utils import hook_save_head_pattern
from utils.rope_features import hook_save_pre_q, hook_save_rot_q, hook_save_rot_k, hook_save_pre_k

LAYER = 0
HEAD = 2
ROT_FEAT = 10

out_arr = []

fwd_hooks = [
    hook_save_pre_q(LAYER, out_arr),
    hook_save_rot_q(LAYER, out_arr),
    hook_save_pre_k(LAYER, out_arr),
    hook_save_rot_k(LAYER, out_arr),
    hook_save_head_pattern(LAYER, HEAD, out_arr),
]
# prompt = task_prompts["Numbers"]
tokens = model.to_tokens(aug_clean_prompts, prepend_bos=False)
# print(text[:100])

model.reset_hooks()
model.set_ungroup_grouped_query_attention(True)
_ = model.run_with_hooks(
    # text[:100],
    tokens,
    fwd_hooks=fwd_hooks,
    )

In [ ]:
from typing import Union, Literal
from utils.rope_features import ablate_rope_features

q_pre = out_arr[0][0] # [n_heads, seq, head_dim]
q_post = out_arr[1][0] # [n_heads, seq, head_dim]
k_pre = out_arr[2][0] # [n_heads, seq, head_dim]
k_post = out_arr[3][0] # [n_heads, seq, head_dim]
pat_before = out_arr[4][0] # [n_heads, seq, head_dim]
# pat_before = pat_before.squeeze(0)  # [n_heads, seq, seq]

# recompute attention
def get_attn(q, k, head_idx, type: Union[Literal['both', 'scores', 'probs']]='probs'):
    """
    q, k: [seq_len, n_heads, head_dim]
    returns: attention matrix for one head [seq_len, seq_len]
    """
    q_ = einops.rearrange(q, "seq head d -> head seq d")
    k_ = einops.rearrange(k, "seq head d -> head d seq")

    scores = torch.matmul(q_, k_) / math.sqrt(head_dim)
    attn = torch.nn.functional.softmax(scores, dim=-1)   # [heads, seq, seq]
    if type == 'both':
        return attn[head_idx], scores[head_idx]
    elif type == 'scores':
        return scores[head_idx]               
    elif type == 'probs':
        return attn[head_idx]                 

variants = ablate_rope_features(q_post, k_post, q_pre, k_pre, ROT_FEAT, type='all_but')

q_post_orig, k_post_orig = variants['orig']
q_abl_zero, k_abl_zero = variants['zero']
q_abl_replace, k_abl_replace = variants['replace']

# aug_prompt = f"The next number in the sequence {task_prompts['Numbers']} is"
words = model.to_str_tokens(aug_clean_prompts[0])   

# 1) get raw scores [seq,seq] for each variant
scores_b  = get_attn(q_post,         k_post,         HEAD, type="scores")
scores_z  = get_attn(q_abl_zero,     k_abl_zero,     HEAD, type="scores")
scores_r  = get_attn(q_abl_replace,  k_abl_replace,  HEAD, type="scores")

# 2) build the causal mask (True = masked/out)
seq_len = scores_b.shape[0]
causal = torch.triu(torch.ones(seq_len, seq_len, device=Q_all.device), 1).bool().to(scores_b.device)

# 3) apply it *in place* to each score tensor
scores_b.masked_fill_(causal_mask, float("-inf"))
scores_z.masked_fill_(causal_mask, float("-inf"))
scores_r.masked_fill_(causal_mask, float("-inf"))

# 4) softmax to get probs
pat_b = torch.softmax(scores_b, dim=-1)
pat_z = torch.softmax(scores_z, dim=-1)
pat_r = torch.softmax(scores_r, dim=-1)

# 5) convert to numpy and wrap in masked arrays
pb = pat_b.cpu().numpy()
pz = pat_z.cpu().numpy()
pr = pat_r.cpu().numpy()

# make the *same* mask in numpy land
np_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)

mb = np.ma.array(pb, mask=np_mask)
mz = np.ma.array(pz, mask=np_mask)
mr = np.ma.array(pr, mask=np_mask)

# 6) plot
cmap = plt.cm.viridis.copy()
cmap.set_bad("lightgray")  
vmax = pb.max()

fig, (ax0,ax1,ax2) = plt.subplots(1,3, figsize=(15,4), sharey=True)
for ax, data, title in zip(
    (ax0, ax1, ax2),
    (mb, mz, mr),
    ("Before Ablation", f"Zeroed RoPE (feat {i})", f"Replaced RoPE (feat {i})")
):
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(words, rotation=90, fontsize=6)
    ax.set_yticklabels(words, fontsize=6)
    ax.set_xlabel("Key Pos")
    ax.set_ylabel("Query Pos")

cbar = fig.colorbar(im, ax=(ax0,ax1,ax2), fraction=0.046, pad=0.04)
cbar.set_label("Attention Score")
plt.show()

i = ROT_FEAT
start = 2*i
width = 2
difs = (q_post - q_pre)[:, start:start+2]   # how much rotation moved those dims 
print("mean |Δ| =", difs.abs().mean().item())
print("mean |q_pre| =", q_pre[:, start:start+2].abs().mean().item())

In [ ]:
extract_rope_frequency_usage(
    model,
    prompt=aug_prompt,
    q_rot=q_abl_zero[:, head, :],
    k_rot=q_abl_zero[:, head, :],
    layer_idx=layer,
    head_idx=head,
    q_or_k="q",
    show_freqs=True
)

# extract_rope_frequency_usage(
#     model,
#     prompt=aug_prompt,
#     q_rot=q_abl_zero[:, head, :],
#     k_rot=q_abl_zero[:, head, :],
#     layer_idx=layer,
#     head_idx=head,
#     q_or_k="k",
#     show_freqs=True
# )

In [ ]:
# next token to predict
nb_str    = "20"       
nb_index  = model.tokenizer.encode(nb_str)[0]

seq_len, n_heads, head_dim = q_pre.shape
i     = ROT_FEAT
start = 2*i
width = 2

variants = ablate_rope_features(q_post, k_post, q_pre, k_pre, ROT_FEAT, type="all_but")

# 4) collect the prob of “nb_str” under each variant
probs = {}
for name, (Qv, Kv) in variants.items():
    # install a hook that overwrites the pattern for (LAYER,HEAD)
    def make_pattern_hook(Qv, Kv, name):
        def hook_pattern(pattern, hook):
            # pattern: [1, heads, seq, seq]
            p = get_attn(Qv, Kv, HEAD)  
            pattern[0, HEAD] = torch.from_numpy(p.cpu().numpy()) # weird fix for a type error
            return pattern
        return hook_pattern

    model.reset_hooks()
    h = (f"blocks.{LAYER}.attn.hook_pattern", make_pattern_hook(Qv, Kv, name))
    logits = model.run_with_hooks(tokens, fwd_hooks=[h], prepend_bos=False)
    p = torch.nn.functional.softmax(logits, dim=-1)[0, -1, nb_index].item()
    probs[name] = p

# 5) bar‐plot them
labels = ["orig", "zero", "replace"]
vals   = [probs[l] for l in labels]
x      = np.arange(len(labels))

plt.figure(figsize=(6,4))
plt.bar(x, vals, color=["C0","C1","C2"])
plt.xticks(x, ["No Ablation", f"Zeroed RoPE {ROT_FEAT}", f"Replaced RoPE {ROT_FEAT}"], rotation=45)
plt.ylabel(f"P(next={nb_str!r})")
plt.title(f"Head {(LAYER,HEAD)} Ablation on RoPE feature {ROT_FEAT}")
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn.functional as F

variants = ablate_rope_features(q_post, k_post, q_pre, k_pre, ROT_FEAT, type="all_but")

# 1) build a causal mask (True where we want to block attention)
causal = torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool, device=device))

def last_query_attn(Qv, Kv):
    # raw scores [seq,seq]
    qh = Qv[:, HEAD, :]     # [seq, d_head]
    kh = Kv[:, HEAD, :]
    S  = (qh @ kh.T) / math.sqrt(d_head)
    # mask out future: keep only lower triangle including diagonal
    S = S.masked_fill(~causal, float("-inf"))
    # softmax
    A = F.softmax(S, dim=-1) # [seq, seq]
    return A[-1]             # return LAST query row [seq]

# 2) compute the three last-query distributions
dist_orig = last_query_attn(*variants["orig"])
dist_zero = last_query_attn(*variants["zero"])
dist_rep  = last_query_attn(*variants["replace"])

str_tokens = model.to_str_tokens(aug_prompt)
# 3) optionally filter to numeric tokens only
NUMERIC_ONLY = False
if NUMERIC_ONLY:
    num_idx = [i for i, tok in enumerate(str_tokens) if tok.isdigit()]
else:
    num_idx = list(range(seq_len))
labels = [str_tokens[i] for i in num_idx]

dist_orig = dist_orig[num_idx].cpu().numpy()
dist_zero = dist_zero[num_idx].cpu().numpy()
dist_rep  = dist_rep[num_idx].cpu().numpy()

# 4) plot
x = np.arange(len(labels))
fig, axes = plt.subplots(1, 3, figsize=(15,4), sharey=True)

for ax, dist, title in zip(axes,
                           [dist_orig, dist_zero, dist_rep],
                           ["Original", f"Zeroed RoPE ({ROT_FEAT})", f"Replaced RoPE ({ROT_FEAT})"]):
    ax.bar(x, dist)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylim(0, dist_orig.max()*1.1)

axes[0].set_ylabel("Attention weight (last query)")
plt.tight_layout()
plt.show()